In [4]:
from pyspark.sql import functions as F
import logging

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("silver.territory.dq")

# ── Read Bronze ───────────────────────────────────────────────────────
df_territory = spark.read.table(
    "lh_adventureworks_bronze.dbo.Sales_SalesTerritory"
)
logger.info(f"Territory rows: {df_territory.count()}")


# ── Select and standardize ────────────────────────────────────────────
df_silver_territory = df_territory.select(
    F.col("TerritoryID"),
    F.col("Name").alias("TerritoryName"),
    F.col("CountryRegionCode"),
    F.col("Group").alias("TerritoryGroup"),
    F.col("SalesYTD").cast("decimal(19,4)"),
    F.col("SalesLastYear").cast("decimal(19,4)"),
    F.col("CostYTD").cast("decimal(19,4)"),
    F.col("CostLastYear").cast("decimal(19,4)"),
    F.col("rowguid"),
    F.col("ModifiedDate")
)

StatementMeta(, 9f66887a-2f34-40c5-8846-870a4b27e50d, 6, Finished, Available, Finished, False)

INFO:silver.territory.dq:Territory rows: 10


In [2]:

# ── DQ checks ─────────────────────────────────────────────────────────
dq_passed = True
dq_results = []

def log_check(name, passed, message):
    global dq_passed
    status = "PASS" if passed else "FAIL"
    log_msg = f"[DQ] [{status}] {name} — {message}"
    if passed:
        logger.info(log_msg)
    else:
        logger.error(log_msg)
        dq_passed = False
    dq_results.append({"check": name, "status": status, "message": message})

actual_rows = df_silver_territory.count()

log_check(
    "Row count",
    actual_rows == df_territory.count(),
    f"Expected {df_territory.count():,}, got {actual_rows:,}"
)

log_check(
    "Null PK — TerritoryID",
    df_silver_territory.filter(F.col("TerritoryID").isNull()).count() == 0,
    f"{df_silver_territory.filter(F.col('TerritoryID').isNull()).count():,} nulls"
)

log_check(
    "Duplicate PK — TerritoryID",
    actual_rows == df_silver_territory.select("TerritoryID").distinct().count(),
    f"{actual_rows - df_silver_territory.select('TerritoryID').distinct().count():,} duplicates"
)

log_check(
    "Null TerritoryName",
    df_silver_territory.filter(F.col("TerritoryName").isNull()).count() == 0,
    "TerritoryName should never be null"
)

failed_checks = [r for r in dq_results if r["status"] == "FAIL"]
if failed_checks:
    failed_names = ", ".join([r["check"] for r in failed_checks])
    raise Exception(
        f"silver.territory DQ failed — {len(failed_checks)} check(s) failed: "
        f"{failed_names}. Pipeline halted."
    )

logger.info(f"[DQ] All checks passed — {actual_rows:,} rows cleared for Silver write.")

StatementMeta(, 9f66887a-2f34-40c5-8846-870a4b27e50d, 4, Finished, Available, Finished, False)

INFO:silver.territory.dq:[DQ] [PASS] Row count — Expected 10, got 10
INFO:silver.territory.dq:[DQ] [PASS] Null PK — TerritoryID — 0 nulls
INFO:silver.territory.dq:[DQ] [PASS] Duplicate PK — TerritoryID — 0 duplicates
INFO:silver.territory.dq:[DQ] [PASS] Null TerritoryName — TerritoryName should never be null
INFO:silver.territory.dq:[DQ] All checks passed — 10 rows cleared for Silver write.


In [3]:
# ── Create schema and write ───────────────────────────────────────────
#spark.sql("CREATE SCHEMA IF NOT EXISTS lh_adventureworks_silver.sales")

df_silver_territory.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("lh_adventureworks_silver.sales.territory")

StatementMeta(, 9f66887a-2f34-40c5-8846-870a4b27e50d, 5, Finished, Available, Finished, False)

silver.sales.territory written — 10 rows verified.


In [ ]:
# ── Verify ────────────────────────────────────────────────────────────
df_verify = spark.read.table("lh_adventureworks_silver.sales.territory")
print(f"silver.sales.territory written — {df_verify.count():,} rows verified.")
